# Presentation figure: outcomes after surgery

One slide figure for "how did patients do?": Engel on the left, ILAE on the right; for each, the share improved vs not improved, then every outcome class. The improved part is in the outcome's colour (**teal Engel, orange ILAE**), the not-improved part grey.

Patients: everyone with that outcome recorded (the second step of the patient flow), so Engel n = 102 and ILAE n = 94. `0_cohort/outcome_overview.ipynb` draws the same charts for the seizure-type cohort the models use (95 / 88). Saved as `results/<version>/presentation/cohort_outcomes.png`. The figure is drawn at slide size (10 x 4.9 in, `cc_stats.plots.SLIDE_SIZE`): place it full width on a 16:9 Google Slides page, below the slide title, and its text keeps its point size.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from cc_stats.outcomes import OUTCOMES
from cc_stats.paths import ANALYSIS_CSV, DATASET
from cc_stats.plots import PAIR_A, PAIR_B, save_presentation_figure, slide_figure

print(f'dataset {DATASET}')
analysis = pd.read_csv(ANALYSIS_CSV, index_col='subject_id')
COLOR = {'engel': PAIR_A, 'ilae': PAIR_B}
NOT_IMPROVED = '#b5b5b5'

In [ ]:
fig = slide_figure()
halves = fig.subfigures(1, 2, wspace=0.04)
for half, (outcome, O) in zip(halves, OUTCOMES.items()):
    recorded = analysis.dropna(subset=[O['improved']])
    n = len(recorded)
    color = COLOR[outcome]
    pie_ax, bar_ax = half.subplots(1, 2, width_ratios=[1, 1.35])
    half.suptitle(f"{O['name']} (n = {n})", fontsize=13, fontweight='bold', color=color)

    # Improved vs not improved
    counts = recorded[O['improved']].map({1: 'improved', 0: 'not improved'}).value_counts().reindex(['improved', 'not improved'])
    pie_ax.pie(counts, colors=[color, NOT_IMPROVED], startangle=90, counterclock=False,
               wedgeprops=dict(edgecolor='white', linewidth=1.5),
               autopct=lambda p: f'{p:.0f}%\n({round(p * n / 100)})', pctdistance=0.62,
               textprops=dict(color='white', fontsize=11, fontweight='bold'))
    pie_ax.legend([f"improved\n({O['improved_label']})", f"not improved\n({O['not_improved_label']})"],
                  loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=1, frameon=False, fontsize=10)

    # Every outcome class
    classes = recorded[O['scale']].astype(int).value_counts().reindex(list(O['classes']), fill_value=0)
    bars = bar_ax.bar([O['classes'][k] for k in classes.index], classes.values, width=0.7,
                      color=[color if k <= O['improved_max'] else NOT_IMPROVED for k in classes.index])
    bar_ax.bar_label(bars, labels=[f'{v}\n{v / n:.0%}' for v in classes.values], fontsize=10, padding=2)
    bar_ax.set_ylim(0, classes.max() * 1.35)
    bar_ax.set_xlabel(f"{O['name']} class")
    bar_ax.set_ylabel('Patients')
    bar_ax.spines[['top', 'right']].set_visible(False)

save_presentation_figure('cohort_outcomes', fig)
plt.show()